# 02 · Custom architectures — §2 [20 marks]

Prototype new layers in `notebooks/scratch/`, move them into `src/edgecnn/models/custom/` once they work, then re-run the cells here — `%autoreload` picks up the edit.

| | |
|---|---|
| **Owner** | Member 2 |
| **Reads** | `configs/stages/models.yaml` (`model_a`, `model_b`) |
| **Writes (official)** | `results/tables/custom_architectures.md`, `param_breakdown.md` |
| **Trains** | nothing — architecture checks only |
| **Report needs** | kernel sizes, filters, FC units; per-layer parameter derivation; total trainable parameters; hardware-aware activation justification |

In [ ]:
# ── Cell 1 · Colab bootstrap ─ does nothing when run locally ───────────────
# On a Colab runtime (in the browser, or from VS Code via the Colab extension) it
# clones REPO at BRANCH, installs it, and on every re-run pulls what you pushed since.
# Locally the package is already installed, so this is skipped. Guide: COLAB.md
import importlib.util
import sys

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    import os
    import subprocess

    def _secret(name, default):  # your Colab Secret if you set one, else the default
        try:
            from google.colab import userdata

            return userdata.get(name) or default
        except Exception:  # not set, or unreadable here (VS Code cannot read Secrets)
            return default

    # The code to run is REPO @ BRANCH. Set your own fork / branch once, in Colab Secrets,
    # as A03_REPO / A03_BRANCH. In VS Code, edit the defaults below for the session and
    # put them back before committing. See COLAB.md.
    REPO = _secret("A03_REPO", "ThejithaR/EN3150-Assignment-03-CNN")
    BRANCH = _secret("A03_BRANCH", "main")
    REPO_DIR = "/content/en3150-a03"

    def _git(*args):  # never prompts: a failure shows as an error, not a hang
        subprocess.run(["git", *args], check=True, env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})

    url = f"https://github.com/{REPO}.git"
    if not os.path.exists(REPO_DIR):
        _git("clone", "--quiet", "--branch", BRANCH, url, REPO_DIR)
    else:  # re-running this cell picks up everything pushed since
        _git("-C", REPO_DIR, "remote", "set-url", "origin", url)
        _git("-C", REPO_DIR, "fetch", "--quiet", "--prune", "origin")
        _git("-C", REPO_DIR, "checkout", "--quiet", "-B", BRANCH, f"origin/{BRANCH}")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", REPO_DIR], check=True)
    if f"{REPO_DIR}/src" not in sys.path:
        sys.path.insert(0, f"{REPO_DIR}/src")  # an editable install is only seen after a restart
    os.chdir(REPO_DIR)
    _git("log", "--oneline", "-1")  # prints the exact commit this run uses

In [ ]:
# ── Cell 2 · Setup ─────────────────────────────────────────────────────────
%load_ext autoreload
%autoreload 2

# MODE decides how much runs and whether anything is written:
#   "synthetic"  generated data, 2 epochs, seconds         - before EuroSAT exists
#   "debug"      5% of EuroSAT, 2 epochs                   - find bugs step by step
#   "official"   the full run - the ONLY mode that writes  - then Restart & Run All
MODE = "synthetic"

from edgecnn.config import load_config
from edgecnn.contracts import paths, schema
import torch

from edgecnn.contracts import MODEL_B_PARAM_BUDGET
from edgecnn.evaluation import count_macs
from edgecnn.models import build_model_from_config, model_input_shape
from edgecnn.models.custom import architecture_tables, layer_table

## Config

In [ ]:
cfg_a = load_config("configs/experiments/model_a__adam.yaml", mode=MODE)
cfg_b = load_config("configs/experiments/model_b__adam.yaml", mode=MODE)
NUM_CLASSES = cfg_a.section("dataset")["expected_num_classes"]
INPUT_SHAPE = model_input_shape(cfg_b)
print(f"{NUM_CLASSES} classes, input {INPUT_SHAPE}, mode={MODE}")

## 1 · Build both models

Always through `build_model_from_config` — the same builder the training notebooks and the benchmark use.

In [ ]:
model_a = build_model_from_config(cfg_a, NUM_CLASSES)
model_b = build_model_from_config(cfg_b, NUM_CLASSES)
model_b

## 2 · Forward-shape check

Both must return raw logits of shape `(batch, 10)` at batch sizes 1 and 2. Rows summing to 1 would mean a softmax was left inside the model.

In [ ]:
for name, model in [("model_a", model_a), ("model_b", model_b)]:
    model.eval()
    with torch.no_grad():
        for batch in (1, 2):
            out = model(torch.randn(batch, *INPUT_SHAPE))
            softmaxed = torch.allclose(out.sum(dim=1), torch.ones(batch), atol=1e-3)
            print(f"{name}  batch {batch}: {tuple(out.shape)}  softmax inside? {softmaxed}")

## 3 · Per-layer parameters — the §2 derivation

Compare every row against your hand calculation.

In [ ]:
layer_table(model_a, INPUT_SHAPE)

In [ ]:
layer_table(model_b, INPUT_SHAPE)

## 4 · Model B's budget: ≤ 100,000 trainable parameters

In [ ]:
trainable = {name: sum(p.numel() for p in m.parameters() if p.requires_grad)
             for name, m in [("model_a", model_a), ("model_b", model_b)]}
print(trainable)
assert trainable["model_b"] <= MODEL_B_PARAM_BUDGET, "over budget - check groups=in_channels and the GAP head"

## 5 · MACs

The number that shows the depthwise-separable saving.

In [ ]:
{name: count_macs(m, INPUT_SHAPE) for name, m in [("model_a", model_a), ("model_b", model_b)]}

## 6 · Activation choice — notes for the report

- ReLU / ReLU6 are a single `max()` — no exponential, no lookup table, no FPU needed.
- sigmoid, tanh, ELU, GELU and Swish each cost a transcendental per activation per inference.
- ReLU6's bounded output keeps int8 quantisation well-scaled — what actually gets a model onto an MCU.
- `hardswish` approximates Swish with add / multiply / clamp only — why MobileNetV3 uses it.

## 7 · Write the §2 tables

In [ ]:
tables = architecture_tables(model_a, model_b, INPUT_SHAPE, write=cfg_a.is_official)

## Official-run checklist

- [ ] `MODE = "official"` in the setup cell
- [ ] Kernel → **Restart & Run All** finished without errors
- [ ] every output is what you expect — no stray warnings or giant prints
- [ ] the per-layer totals match your hand derivation in the report
- [ ] `pytest tests/contracts` is green
- [ ] commit this notebook **with its outputs**, together with the files it wrote

In [ ]:
# ── Colab only: send this run's results back to GitHub ─────────────────────
# Does nothing locally, or in synthetic / debug mode. Guide: COLAB.md
if IN_COLAB and MODE == "official":
    from edgecnn.utils import push_results

    push_results(
        [paths.CUSTOM_ARCHITECTURES_TABLE, paths.PARAM_BREAKDOWN_TABLE],
        "Add architecture tables (Colab)",
    )